# Utils — Helper Functions

Reusable utilities for the ANTHOR project.
All paths use **Google Drive**.

## 1. Mount Google Drive (MANDATORY)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Paths & Config

In [ ]:
import os

DRIVE_BASE = "/content/drive/MyDrive/ANTHOR"

PATHS = {
    "tokenizer": f"{DRIVE_BASE}/tokenizer/tokenizer.json",
    "model_config": f"{DRIVE_BASE}/model_config.json",
    "dataset_meta": f"{DRIVE_BASE}/dataset_meta.json",
    "checkpoints": f"{DRIVE_BASE}/checkpoints",
    "data_shards": f"{DRIVE_BASE}/data_shards",
    "data_val": f"{DRIVE_BASE}/data_shards_val",
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"{name:15} → {path}")

## 3. Load latest checkpoint

In [ ]:
def get_latest_checkpoint(checkpoint_dir):
    """Return (path, step) of latest checkpoint."""
    ckpts = sorted([f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_") and f.endswith(".pt")])
    if not ckpts:
        return None, 0
    
    latest = ckpts[-1]
    step = int(latest.replace("checkpoint_", "").replace(".pt", ""))
    return os.path.join(checkpoint_dir, latest), step

ckpt_path, step = get_latest_checkpoint(PATHS["checkpoints"])
if ckpt_path:
    print(f"Latest checkpoint: {ckpt_path} (step={step})")
else:
    print("No checkpoint found.")

## 4. Sample generation

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=1.0, top_k=50, device="cuda"):
    """Generate text from prompt."""
    model.eval()
    
    # Encode prompt
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Trim to max context
            ctx = input_ids[:, -model.config.max_seq_len:]
            logits, _ = model(ctx)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
    
    return tokenizer.decode(input_ids[0].tolist())

---

✅ These utilities are used across all notebooks:
- `PATHS` — consistent Google Drive paths
- `get_latest_checkpoint()` — resume training
- `generate()` — text generation